# Section 4 — integer rounding starter

**Alan: change integer rounding only.** Destination, timing, capacity methodology, target weights, common benchmark, accounting and metric stay fixed. Consume already selected exact contracts and post-capacity fractional targets with supplied integer ceilings.

[Part 2 Issue #6](https://github.com/JacopoMichelacci/Futures_Class/issues/6), Version 8 §§59–62. This checkout is based only on Section 0 Phase A. The separate local Issue #6 implementation is deliberately not imported or copied here; use TODOs until its reviewed module is available on the shared base.

Run all cells from a fresh Python kernel using the project's existing uv environment. These notebooks use only the shared synthetic fixture and standard-library Python, with no online access. Outputs are deliberately cleared before sharing.

**Part 2 responsibility:** each teammate implements their own reusable engine and substantive tests, opens their own issue-linked PR, and reviews teammate PRs. This starter supplies inputs and interface examples, not the strand solution. Keep engine logic in `src/` and tests in `tests/unit/`. No empirical conclusions or preferred-method selection belong here. Do not consume another strand's preferred results.

See [Section 0 interfaces](../docs/section0_phase_a.md) and [fixture provenance](../data/fixtures/README.md). Phase B still needs sourced first-notice metadata, verified exchange/shortened-session times, historical benchmark/baseline quantities, production tables, the common 100-date/70% gate and AUM/capacity preflight. The synthetic calendar is not a real CBOT calendar. Daily return Tracking RMSE in basis points remains the sole ranking metric for the later historical comparison.

In [ ]:
from pathlib import Path
from decimal import Decimal
from pprint import pprint
import sys

# Start the kernel in this repository or the notebook's own folder.
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "config/project.yaml").is_file()
             and (p / "data/fixtures/phase_a.json").is_file()
             and (p / "src/__init__.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the kernel in the repository or this notebook's folder.")
sys.path.insert(0, str(ROOT))
import src
assert Path(src.__file__).resolve().parent == ROOT / "src", "Restart the kernel for this checkout."
from src.validation.schemas import load_project_config, HOLDINGS_COLUMNS
from src.data.fixtures import load_fixture

config = load_project_config(ROOT / "config/project.yaml")
f = load_fixture(ROOT / "data/fixtures/phase_a.json",
                 config["max_liquidity_source_age_sessions"])
return_date = f.raw["return_dates"][0]
context = f.calendar.decision(return_date)
print("SYNTHETIC FIXTURES ONLY", config["roots"], "return interval ends", return_date)


### Inspect the supplied fractional targets

Each row preserves commodity, exact contract, target weight/notional, post-capacity fractional quantity, dollar contract notional and integer ceiling. The fixture supplies these values as a separate hand-input case. Do not recompute destination or capacity from them. Input availability must be known by the preceding decision cut-off, separately from later settlement marks used for valuation.

In [ ]:
targets = f.raw["post_capacity_targets"]
pprint(targets)
for row in targets:
    assert row["contract"] in f.catalogue
    assert f.catalogue[row["contract"]].identity.root == row["commodity"]
    quantity = Decimal(row["post_capacity_fractional_contracts"])
    ceiling = Decimal(row["capacity_limit_contracts"])
    assert 0 <= quantity <= ceiling and ceiling == ceiling.to_integral_value()
print("Supplied exact-contract inputs are within their integer ceilings.")


### Four planned rounding rules

| Rule | Required behaviour |
| --- | --- |
| Nearest (baseline) | Independent nearest integer; **approved half-up** ties: 0.5 → 1, 2.5 → 3, 3.5 → 4. This resolves an otherwise unspecified deterministic boundary, not a research finding. |
| Toward zero | Floor each non-negative fractional quantity. |
| Away from zero | Smallest integer at least as large as each quantity; zero stays zero. |
| Residual greedy | Start from floor, add the feasible contract with greatest strictly positive reduction in squared dollar-notional deviation from post-capacity quantities; stop without positive improvement. Exact ties use ZC, ZS, ZW, then earlier maturity. |

All rules preserve the same contracts, targets and ceilings. Non-negative integer results may not exceed supplied integer ceilings. Greedy's construction objective is not a ranking metric; only the later shared daily return Tracking RMSE ranks historical implementations. In particular, a greedy addition at an exact half gives zero improvement, so half-up does not change greedy's stopping rule.

**TODO Alan:** supply `round_nearest`, `round_toward_zero`, `round_away_from_zero`, `round_residual_greedy` and `build_rounded_portfolio` from the separately reviewed Issue #6 contribution. Test hand examples, exact halves, zero/binding ceilings, unchanged contract/target sets, deterministic root/maturity ties and monotonic greedy improvement. Open the rounding-only PR separately from this onboarding contribution.

In [ ]:
# This starter intentionally has no import from a strand implementation.
rounding_engine = None
rounded_quantities = None
# Eventual shape: {exact_contract: non_negative_integer}, for every supplied target.
expected_contracts = [row["contract"] for row in targets]
print("TODO integer quantities for:", expected_contracts)


### Future rounded holdings

Pass the implemented integer quantity, unchanged contract/commodity and supplied ceiling to `Holding`, with truthful `formed_at`, `inputs_as_of` and the shared `decision_cutoff`. Retain target notionals and dollar contract notionals as descriptive fields. Preserve zero positions explicitly. The inactive adapter below illustrates the common accounting boundary without copying the rounding engine.

### Integration boundary: future engine output → shared accounting

The TODO below is intentionally inactive. Your engine must produce canonical `Holding` rows for the baseline and all three alternatives on the same fixture dates, plus truthful decision-validity flags. Each row's `date` is the end of `(t-1,t]`; quantities and inputs are fixed by the preceding cut-off. Preserve exact contracts and explicit zero positions. Decision timestamps cannot be invented from later valuation marks.

Use the supplied fixture benchmark once and call the shared engine once for all compared methods. Missing required settlements invalidate the common date; never fill them with zero or splice maturities. One fixed AUM divides every portfolio's P&L. This adapter is for the synthetic accounting example only; the hand-authored benchmark quantities are not a constructed 60-session portfolio. Phase B must supply the actual common benchmark and frozen comparison sample.

In [ ]:
from src.portfolio.holdings import HoldingsBook
from src.metrics.portfolio_pnl import portfolio_pnl
from src.metrics.tracking_rmse_bps import tracking_rmse_bps

future_holding_rows = None  # TODO: baseline + three alternatives, produced by your engine.
future_decision_validity = None  # TODO: {(date, strategy): bool}, including the benchmark.
print("Canonical fields:", HOLDINGS_COLUMNS)
if future_holding_rows is None or future_decision_validity is None:
    print("TODO adapter: no strategy implemented or evaluated by this starter.")
else:
    benchmark_rows = [r for r in f.holdings.rows if r.strategy == f.raw["benchmark_strategy"]]
    strategies = [f.raw["benchmark_strategy"]] + sorted({r.strategy for r in future_holding_rows})
    assert len(strategies) == len(set(strategies)) == 5, "Provide baseline and three distinct alternatives."
    book = HoldingsBook(benchmark_rows + list(future_holding_rows), f.calendar, f.catalogue)
    daily = portfolio_pnl(book, f.observations, f.raw["return_dates"], strategies,
                          f.raw["fixed_aum"], f.raw["snapshot_end_exclusive"],
                          decision_validity=future_decision_validity)
    pprint(tracking_rmse_bps(daily))  # Synthetic arithmetic check; no preferred-method selection.
